**Exercise 1: Order Status Lookup Tool**

Step 1 — Setup


In [1]:
%pip install -q -U langchain-core langchain-groq pydantic

In [2]:
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=getpass("Enter your Groq API key: ")
)

print("LLM initialized successfully!")

Enter your Groq API key: ··········
LLM initialized successfully!


Step 2 — Create the order lookup tool

In [3]:
@tool
def lookup_order_status(order_id: str) -> str:
    """Look up the shipping status of an order using its order ID."""

    orders = {
        "ORD-5621": "Shipped and expected to arrive tomorrow.",
        "ORD-8743": "Your order is being packed for dispatch.",
        "ORD-3196": "Delivered successfully."
    }

    return orders.get(order_id, "Order ID not found.")

print("Order lookup tool created!")

Order lookup tool created!


Step 3 — Bind the tool and inspect the model's suggestion

In [4]:
tools = [lookup_order_status]
llm_with_tools = llm.bind_tools(tools)

user_query = "Where is my order ORD-5621?"

response = llm_with_tools.invoke(user_query)

print("Model's Suggested Tool Calls:")
print(response.tool_calls)

Model's Suggested Tool Calls:
[{'name': 'lookup_order_status', 'args': {'order_id': 'ORD-5621'}, 'id': 'fc_f7fbb7ff-d49a-4dff-9c08-6ad212a838e4', 'type': 'tool_call'}]


Step 4 — Execute the tool manually

In [5]:
tool_messages = []

for tool_call in response.tool_calls:
    if tool_call["name"] == "lookup_order_status":
        result = lookup_order_status.invoke(tool_call["args"])

        print("Manually Executed Tool Result:")
        print(result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

Manually Executed Tool Result:
Shipped and expected to arrive tomorrow.


Step 5 — Generate the final answer

In [6]:
final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("Final Answer:")
print(final_response.content)

Final Answer:
Your order **ORD‑5621** has already shipped. It’s on its way and is expected to arrive tomorrow. Let me know if there’s anything else I can help with!


**Exercise 2: Loan EMI Calculator Tool.**

Step 1 — Define the validation schema

In [7]:
from pydantic import BaseModel, Field

class LoanEMIInput(BaseModel):
    principal: float = Field(gt=0, description="Loan amount in rupees")
    annual_rate: float = Field(
        ge=0, le=30,
        description="Annual interest rate as a percentage"
    )
    tenure_months: int = Field(
        gt=0, description="Loan tenure in months"
    )

Step 2 — Create the EMI calculator tool

In [8]:
from langchain_core.tools import StructuredTool

def calculate_emi(
    principal: float,
    annual_rate: float,
    tenure_months: int
) -> str:
    """Calculate monthly EMI using loan amount, annual interest rate, and tenure."""

    monthly_rate = annual_rate / (12 * 100)

    if monthly_rate == 0:
        emi = principal / tenure_months
    else:
        emi = (
            principal
            * monthly_rate
            * (1 + monthly_rate) ** tenure_months
            / ((1 + monthly_rate) ** tenure_months - 1)
        )

    return f"Monthly EMI: ₹{emi:.2f}"

emi_tool = StructuredTool.from_function(
    func=calculate_emi,
    name="calculate_loan_emi",
    description=(
        "Calculate monthly loan EMI. "
        "Requires principal in rupees, annual_rate as a percentage, "
        "and tenure_months."
    ),
    args_schema=LoanEMIInput
)

print("EMI calculator tool created successfully!")

EMI calculator tool created successfully!


Step 3 — Bind the tool to the model

In [9]:
tools = [emi_tool]
llm_with_tools = llm.bind_tools(tools)

user_query = "What's my EMI for a ₹5 lakh loan at 9% annual interest for 24 months?"

response = llm_with_tools.invoke(user_query)

print("Model's Suggested Tool Calls:")
print(response.tool_calls)

Model's Suggested Tool Calls:
[{'name': 'calculate_loan_emi', 'args': {'annual_rate': 9, 'principal': 500000, 'tenure_months': 24}, 'id': 'fc_24d099f7-c11a-41c4-a817-dc524b65185e', 'type': 'tool_call'}]


Step 4 — Manually execute the suggested tool call

In [10]:
from langchain_core.messages import ToolMessage, HumanMessage

tool_messages = []

for tool_call in response.tool_calls:
    if tool_call["name"] == "calculate_loan_emi":
        result = emi_tool.invoke(tool_call["args"])

        print("Manually Executed Tool Result:")
        print(result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

Manually Executed Tool Result:
Monthly EMI: ₹22842.37


Step 5 — Generate the final natural-language answer

In [11]:
final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("Final Answer:")
print(final_response.content)

Final Answer:
Your monthly EMI would be **₹ 22,842.37**.

**Quick recap of the calculation**

- **Principal (P):** ₹ 5,00,000  
- **Annual interest rate (r):** 9 % → monthly rate = 9 % / 12 = 0.75 % (0.0075)  
- **Tenure (n):** 24 months  

EMI formula:  

\[
EMI = P \times \frac{r(1+r)^n}{(1+r)^n-1}
\]

Plugging in the numbers:

\[
EMI = 5{,}00{,}000 \times \frac{0.0075(1+0.0075)^{24}}{(1+0.0075)^{24}-1} \approx ₹\,22,842.37
\]

So each month for 24 months, you’ll pay about ₹ 22,842.37. Let me know if you’d like a more detailed breakdown or if you have any other questions!


Step 6 — Test strict validation

In [12]:
from pydantic import ValidationError

test_cases = [
    {"principal": 0, "annual_rate": 9, "tenure_months": 24},
    {"principal": 500000, "annual_rate": 35, "tenure_months": 24},
    {"principal": 500000, "annual_rate": 9, "tenure_months": 0}
]

for case in test_cases:
    try:
        LoanEMIInput.model_validate(case)
        print("Accepted:", case)
    except ValidationError:
        print("Rejected invalid input:", case)

Rejected invalid input: {'principal': 0, 'annual_rate': 9, 'tenure_months': 24}
Rejected invalid input: {'principal': 500000, 'annual_rate': 35, 'tenure_months': 24}
Rejected invalid input: {'principal': 500000, 'annual_rate': 9, 'tenure_months': 0}


**Exercise 3: Unit Converter Toolkit (Multiple Tools).**

Step 1 — Create both conversion tools

In [13]:
from langchain_core.tools import tool

@tool
def convert_temperature(value: float, conversion: str) -> str:
    """Convert temperature between Celsius and Fahrenheit.
    Use conversion='C_to_F' for Celsius to Fahrenheit,
    or conversion='F_to_C' for Fahrenheit to Celsius.
    """
    if conversion == "C_to_F":
        result = (value * 9 / 5) + 32
        return f"{value}°C = {result:.2f}°F"

    elif conversion == "F_to_C":
        result = (value - 32) * 5 / 9
        return f"{value}°F = {result:.2f}°C"

    return "Invalid conversion. Use C_to_F or F_to_C."


@tool
def convert_distance(value: float, conversion: str) -> str:
    """Convert distance between kilometers and miles.
    Use conversion='km_to_miles' for kilometers to miles,
    or conversion='miles_to_km' for miles to kilometers.
    """
    if conversion == "km_to_miles":
        result = value * 0.621371
        return f"{value} km = {result:.2f} miles"

    elif conversion == "miles_to_km":
        result = value / 0.621371
        return f"{value} miles = {result:.2f} km"

    return "Invalid conversion. Use km_to_miles or miles_to_km."


print("Both conversion tools created successfully!")

Both conversion tools created successfully!


Step 2 — Bind both tools to the model

In [14]:
tools = [convert_temperature, convert_distance]
llm_with_tools = llm.bind_tools(tools)

user_query = "Convert 100 km to miles."

response = llm_with_tools.invoke(user_query)

print("Model's Suggested Tool Calls:")
print(response.tool_calls)

Model's Suggested Tool Calls:
[{'name': 'convert_distance', 'args': {'conversion': 'km_to_miles', 'value': 100}, 'id': 'fc_2c6a654e-f715-4e28-bc58-18d95e916614', 'type': 'tool_call'}]


Step 3 — Execute the suggested tool call manually

In [15]:
from langchain_core.messages import ToolMessage, HumanMessage

tool_messages = []

for tool_call in response.tool_calls:
    selected_tool = next(
        (t for t in tools if t.name == tool_call["name"]),
        None
    )

    if selected_tool:
        result = selected_tool.invoke(tool_call["args"])

        print(f"\nSelected tool: {tool_call['name']}")
        print("Tool result:", result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )


Selected tool: convert_distance
Tool result: 100.0 km = 62.14 miles


Step 4 — Get the final answer

In [16]:
final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("Final Answer:")
print(final_response.content)

Final Answer:
100 km is approximately **62.14 miles**.


**Exercise 4 — Weather-Based Outfit Recommendation**

Create the tools

In [17]:
import requests
import json
from langchain_core.tools import tool

@tool
def get_weather(city: str) -> str:
    """Get current temperature in Celsius and weather conditions for a city."""
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "en", "format": "json"},
        timeout=15
    )
    geo_response.raise_for_status()
    places = geo_response.json().get("results", [])

    if not places:
        return json.dumps({"error": f"City not found: {city}"})

    place = places[0]
    weather_response = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,apparent_temperature,weather_code",
            "timezone": "auto"
        },
        timeout=15
    )
    weather_response.raise_for_status()
    current = weather_response.json().get("current", {})

    code = current.get("weather_code")
    descriptions = {
        0: "clear sky",
        1: "mainly clear",
        2: "partly cloudy",
        3: "overcast",
        45: "foggy",
        48: "foggy",
        51: "light drizzle",
        61: "rain",
        63: "moderate rain",
        65: "heavy rain",
        71: "snow",
        80: "rain showers",
        95: "thunderstorm"
    }

    data = {
        "city": place["name"],
        "temperature": current.get("temperature_2m"),
        "conditions": descriptions.get(code, f"weather code {code}"),
        "observed_at": current.get("time")
    }
    return json.dumps(data)


@tool
def recommend_outfit(temperature: float, conditions: str) -> str:
    """Recommend an outfit using temperature in Celsius and weather conditions."""
    condition = conditions.lower()

    if "rain" in condition or "drizzle" in condition or "thunderstorm" in condition:
        return "Carry an umbrella or raincoat and wear water-resistant footwear."
    elif temperature < 12:
        return "Wear a warm jacket, full-length trousers, and closed shoes."
    elif temperature < 22:
        return "Wear a light jacket or long-sleeved top with comfortable trousers."
    elif temperature < 30:
        return "Wear a breathable T-shirt and lightweight trousers."
    else:
        return "Wear light, breathable clothing and consider a hat and sunglasses."

First model call: request weather

In [18]:
from langchain_core.messages import HumanMessage, ToolMessage

tools = [get_weather, recommend_outfit]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

user_query = "What should I wear today in Mumbai? First check the current weather."

response1 = llm_with_tools.invoke(user_query)

print("ROUND 1 — MODEL TOOL CALLS")
print(response1.tool_calls)

ROUND 1 — MODEL TOOL CALLS
[{'name': 'get_weather', 'args': {'city': 'Mumbai'}, 'id': 'fc_fd444f50-686d-4264-a3c7-fb707f1fc7bb', 'type': 'tool_call'}]


Execute the weather tool and feed the result back

In [19]:
weather_messages = []
weather_result = None

for call in response1.tool_calls:
    if call["name"] == "get_weather":
        weather_result = get_weather.invoke(call["args"])
        print("FETCHED WEATHER:")
        print(weather_result)

        weather_messages.append(
            ToolMessage(
                content=str(weather_result),
                tool_call_id=call["id"]
            )
        )

if weather_result is None:
    raise RuntimeError("The model did not call get_weather. Rerun the previous cell.")

FETCHED WEATHER:
{"city": "Mumbai", "temperature": 29.5, "conditions": "clear sky", "observed_at": "2026-10-24T20:30"}


Second model call: request outfit recommendation

In [20]:
response2 = llm_with_tools.invoke([
    HumanMessage(content=user_query),
    response1,
    *weather_messages,
    HumanMessage(
        content=(
            "Now call recommend_outfit using the exact temperature and "
            "conditions from the fetched weather result above. "
            "Do not invent or replace the weather values."
        )
    )
])

print("ROUND 2 — MODEL TOOL CALLS")
print(response2.tool_calls)

ROUND 2 — MODEL TOOL CALLS
[{'name': 'recommend_outfit', 'args': {'conditions': 'clear sky', 'temperature': 29.5}, 'id': 'fc_e2faff49-b46f-4045-9816-9b5d3c81564a', 'type': 'tool_call'}]


Execute the outfit tool and get the final answe

In [21]:
outfit_messages = []
outfit_result = None

for call in response2.tool_calls:
    if call["name"] == "recommend_outfit":
        outfit_result = recommend_outfit.invoke(call["args"])
        print("OUTFIT TOOL RESULT:", outfit_result)

        outfit_messages.append(
            ToolMessage(
                content=str(outfit_result),
                tool_call_id=call["id"]
            )
        )

if outfit_result is None:
    raise RuntimeError("The model did not call recommend_outfit. Rerun the previous cell.")

final_response = llm.invoke([
    HumanMessage(content=user_query),
    response1,
    *weather_messages,
    response2,
    *outfit_messages
])

print("\nFINAL OUTFIT RECOMMENDATION")
print(final_response.content)

OUTFIT TOOL RESULT: Wear a breathable T-shirt and lightweight trousers.

FINAL OUTFIT RECOMMENDATION
The current conditions in Mumbai are **clear sky** with a temperature around **29.5 °C**.  
A light, breathable outfit will keep you comfortable:

- **Top**: A cotton or moisture‑wicking T‑shirt (or a short‑sleeve linen shirt if you prefer a bit more coverage).  
- **Bottom**: Lightweight trousers or chinos, or even a pair of comfortable shorts if you’re going to be outdoors all day.  
- **Footwear**: Sandals, canvas shoes, or breathable sneakers.  
- **Extras**: Sunglasses, a wide‑brim hat, and a light jacket or shawl if you’ll be in air‑conditioned indoor spaces.  
- **Hydration**: Carry a reusable water bottle to stay hydrated in the heat.

Enjoy your day in Mumbai!


**Exercise 5 — Async Product Stock Checker**

Cell 1 — Define the tool

In [22]:
import asyncio
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool

class ProductStockInput(BaseModel):
    product_id: str = Field(description="Product ID to check")


class ProductStockChecker(BaseTool):
    name: str = "check_product_stock"
    description: str = "Check whether a product is in stock using its product ID."
    args_schema: type[BaseModel] = ProductStockInput

    def _run(self, product_id: str) -> str:
        """Synchronously check product stock."""
        inventory = {
            "P1023": 12,
            "P2045": 0,
            "P3018": 7
        }

        if product_id not in inventory:
            return f"Product {product_id} was not found."

        quantity = inventory[product_id]
        if quantity > 0:
            return f"Product {product_id} is in stock. Quantity: {quantity}."
        return f"Product {product_id} is out of stock."

    async def _arun(self, product_id: str) -> str:
        """Asynchronously check product stock."""
        await asyncio.sleep(0)
        return self._run(product_id)


stock_tool = ProductStockChecker()
print("Product stock tool created!")

Product stock tool created!


Cell 2 — Manual synchronous and asynchronous calls

In [23]:
product_id = "P1023"

sync_result = stock_tool.invoke({"product_id": product_id})
print("SYNC RESULT:", sync_result)

async_result = await stock_tool.ainvoke({"product_id": product_id})
print("ASYNC RESULT:", async_result)

print("Results match:", sync_result == async_result)

SYNC RESULT: Product P1023 is in stock. Quantity: 12.
ASYNC RESULT: Product P1023 is in stock. Quantity: 12.
Results match: True


Cell 3 — Bind the tool to the model

In [24]:
tools = [stock_tool]
llm_with_tools = llm.bind_tools(tools)

user_query = "Is product P1023 in stock?"
response = llm_with_tools.invoke(user_query)

print("MODEL'S SUGGESTED TOOL CALLS:")
print(response.tool_calls)

MODEL'S SUGGESTED TOOL CALLS:
[{'name': 'check_product_stock', 'args': {'product_id': 'P1023'}, 'id': 'fc_4e63c5ec-a109-4924-a9ef-341093916b59', 'type': 'tool_call'}]


Cell 4 — Execute the tool and generate the final answer

In [25]:
tool_messages = []

for call in response.tool_calls:
    if call["name"] == "check_product_stock":
        result = stock_tool.invoke(call["args"])
        print("MANUAL TOOL RESULT:", result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("FINAL ANSWER:")
print(final_response.content)

MANUAL TOOL RESULT: Product P1023 is in stock. Quantity: 12.
FINAL ANSWER:
Yes—Product P1023 is currently in stock. We have **12 units** available. Let me know if you’d like to place an order or need any other details!


**Exercise 6 — Expense Splitter Tool**

Cell 1 — Create the tool

In [26]:
from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class ExpenseSplitInput(BaseModel):
    total_amount: float = Field(gt=0, description="Total bill amount in rupees")
    num_people: int = Field(gt=0, description="Number of people sharing the bill")

def split_expense(total_amount: float, num_people: int) -> str:
    """Split a total bill equally among a positive number of people."""
    amount_per_person = total_amount / num_people
    return (
        f"Total bill: ₹{total_amount:.2f}. "
        f"People: {num_people}. "
        f"Each person owes: ₹{amount_per_person:.2f}."
    )

expense_tool = StructuredTool.from_function(
    func=split_expense,
    name="split_expense",
    description="Split a bill equally among a given number of people.",
    args_schema=ExpenseSplitInput
)

Cell 2 — Call manually without the model

In [27]:
total_amount = 2400
num_people = 4

manual_result = expense_tool.invoke({
    "total_amount": total_amount,
    "num_people": num_people
})

print("MANUAL RESULT:")
print(manual_result)

MANUAL RESULT:
Total bill: ₹2400.00. People: 4. Each person owes: ₹600.00.


Cell 3 — Bind the tool and inspect the model's call

In [28]:
tools = [expense_tool]
llm_with_tools = llm.bind_tools(tools)

user_query = "Split ₹2400 equally between 4 friends."
response = llm_with_tools.invoke(user_query)

print("MODEL'S SUGGESTED TOOL CALLS:")
print(response.tool_calls)

MODEL'S SUGGESTED TOOL CALLS:
[{'name': 'split_expense', 'args': {'num_people': 4, 'total_amount': 2400}, 'id': 'fc_821f0919-cd5f-4110-aa97-d2b38f4d1339', 'type': 'tool_call'}]


Cell 4 — Execute the suggested call and compare results

In [29]:
tool_messages = []
bound_result = None

for call in response.tool_calls:
    if call["name"] == "split_expense":
        bound_result = expense_tool.invoke(call["args"])
        print("BOUND-FLOW TOOL RESULT:")
        print(bound_result)

        tool_messages.append(
            ToolMessage(
                content=str(bound_result),
                tool_call_id=call["id"]
            )
        )

if bound_result is None:
    raise RuntimeError("The model did not call split_expense. Rerun the previous cell.")

print("\nMANUAL RESULT:", manual_result)
print("BOUND RESULT:", bound_result)
print("RESULTS MATCH:", manual_result == bound_result)

final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("\nFINAL ANSWER:")
print(final_response.content)

BOUND-FLOW TOOL RESULT:
Total bill: ₹2400.00. People: 4. Each person owes: ₹600.00.

MANUAL RESULT: Total bill: ₹2400.00. People: 4. Each person owes: ₹600.00.
BOUND RESULT: Total bill: ₹2400.00. People: 4. Each person owes: ₹600.00.
RESULTS MATCH: True

FINAL ANSWER:
Each friend should pay ₹600.


**Exercise 7 — Trip Cost Estimator**

Cell 1 — Define the tools

In [30]:
from langchain_core.tools import tool

@tool
def get_flight_cost(origin: str, destination: str) -> str:
    """Estimate round-trip flight cost in rupees."""
    fares = {
        ("Delhi", "Goa"): 8000,
        ("Mumbai", "Goa"): 5000,
        ("Delhi", "Mumbai"): 6000
    }
    cost = fares.get((origin.title(), destination.title()))

    if cost is None:
        cost = 9000  # Demonstration fallback estimate

    return f"Estimated round-trip flights: ₹{cost}."


@tool
def get_hotel_cost(city: str, nights: int) -> str:
    """Estimate total hotel cost in rupees for the specified number of nights."""
    rates = {"Goa": 3000, "Mumbai": 4000, "Delhi": 3500}
    nightly_rate = rates.get(city.title(), 3200)

    return (
        f"Estimated hotel cost in {city} for {nights} nights: "
        f"₹{nightly_rate * nights}."
    )


@tool
def get_food_budget(city: str, days: int) -> str:
    """Estimate total food budget in rupees for the specified number of days."""
    budgets = {"Goa": 1000, "Mumbai": 1200, "Delhi": 900}
    daily_budget = budgets.get(city.title(), 1000)

    return (
        f"Estimated food budget in {city} for {days} days: "
        f"₹{daily_budget * days}."
    )

Cell 2 — Bind all three tools and inspect calls

In [31]:
tools = [get_flight_cost, get_hotel_cost, get_food_budget]
tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

user_query = """
I'm planning a 3-night trip from Delhi to Goa.
Estimate my round-trip flights, hotel for 3 nights, and food for 4 days.
Call all three tools: get_flight_cost, get_hotel_cost, and get_food_budget.
"""

response = llm_with_tools.invoke(user_query)

print("MODEL'S SUGGESTED TOOL CALLS:")
for call in response.tool_calls:
    print(call)

MODEL'S SUGGESTED TOOL CALLS:
{'name': 'get_flight_cost', 'args': {'destination': 'Goa', 'origin': 'Delhi'}, 'id': 'fc_5b8d3847-534b-4e6c-927e-a648cc90b7c3', 'type': 'tool_call'}


Cell 3 — Execute the calls and return results

In [32]:
tool_messages = []
results = {}

for call in response.tool_calls:
    selected_tool = tool_map.get(call["name"])

    if selected_tool:
        result = selected_tool.invoke(call["args"])
        results[call["name"]] = result

        print(f"\n{call['name']} RESULT:")
        print(result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

required_tools = {
    "get_flight_cost",
    "get_hotel_cost",
    "get_food_budget"
}

missing = required_tools - set(results)

if missing:
    raise RuntimeError(
        f"The model omitted {missing}. Rerun the model-call cell and check its tool_calls."
    )

final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("\nFINAL TRIP COST ESTIMATE:")
print(final_response.content)


get_flight_cost RESULT:
Estimated round-trip flights: ₹8000.


RuntimeError: The model omitted {'get_hotel_cost', 'get_food_budget'}. Rerun the model-call cell and check its tool_calls.

**Exercise 8 — Health Metrics Toolkit (BMI + Calorie Needs)**

Cell 1 — Define schemas and calculations

In [33]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class BMIInput(BaseModel):
    weight_kg: float = Field(gt=0, description="Weight in kilograms")
    height_cm: float = Field(gt=0, description="Height in centimeters")


class CalorieInput(BaseModel):
    weight_kg: float = Field(gt=0, description="Weight in kilograms")
    height_cm: float = Field(gt=0, description="Height in centimeters")
    age: int = Field(gt=0, description="Age in years")
    activity_level: str = Field(
        description="One of sedentary, lightly_active, moderately_active, very_active"
    )


def calculate_bmi(weight_kg: float, height_cm: float) -> str:
    """Calculate BMI from weight in kilograms and height in centimeters."""
    height_m = height_cm / 100
    bmi = weight_kg / (height_m ** 2)
    return f"BMI: {bmi:.2f}"


def calculate_daily_calories(
    weight_kg: float,
    height_cm: float,
    age: int,
    activity_level: str
) -> str:
    """Estimate daily calories using an activity multiplier and the male BMR equation."""
    bmr = (10 * weight_kg) + (6.25 * height_cm) - (5 * age) + 5

    multipliers = {
        "sedentary": 1.2,
        "lightly_active": 1.375,
        "moderately_active": 1.55,
        "very_active": 1.725
    }

    level = activity_level.lower().replace(" ", "_")
    if level not in multipliers:
        return "Invalid activity level. Use sedentary, lightly_active, moderately_active, or very_active."

    calories = bmr * multipliers[level]
    return f"Estimated daily calories: {calories:.0f} kcal."

Cell 2 — Create the tools and toolkit

In [34]:
bmi_tool = StructuredTool.from_function(
    func=calculate_bmi,
    name="calculate_bmi",
    description="Calculate BMI using weight in kg and height in cm.",
    args_schema=BMIInput
)

calorie_tool = StructuredTool.from_function(
    func=calculate_daily_calories,
    name="calculate_daily_calories",
    description="Estimate daily calorie needs from weight, height, age, and activity level.",
    args_schema=CalorieInput
)


class HealthToolkit:
    def get_tools(self):
        return [bmi_tool, calorie_tool]


health_toolkit = HealthToolkit()
tools = health_toolkit.get_tools()
tool_map = {t.name: t for t in tools}

print("TOOLKIT TOOLS:")
print([t.name for t in tools])

TOOLKIT TOOLS:
['calculate_bmi', 'calculate_daily_calories']


Cell 3 — Ask the model to use both tools

In [35]:
llm_with_tools = llm.bind_tools(tools)

user_query = """
I'm 70 kg, 175 cm tall, 25 years old, and moderately active.
Calculate both my BMI and estimated daily calorie needs.
Use calculate_bmi and calculate_daily_calories.
"""

response = llm_with_tools.invoke(user_query)

print("MODEL'S SUGGESTED TOOL CALLS:")
for call in response.tool_calls:
    print(call)

MODEL'S SUGGESTED TOOL CALLS:
{'name': 'calculate_bmi', 'args': {'height_cm': 175, 'weight_kg': 70}, 'id': 'fc_a77c7327-8b0d-4f55-8481-5f5f6a38c9a9', 'type': 'tool_call'}


Cell 4 — Execute both tools and get the final answe

In [36]:
tool_messages = []
results = {}

for call in response.tool_calls:
    selected_tool = tool_map.get(call["name"])

    if selected_tool:
        result = selected_tool.invoke(call["args"])
        results[call["name"]] = result

        print(f"\n{call['name']} RESULT:")
        print(result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

required_tools = {"calculate_bmi", "calculate_daily_calories"}
missing = required_tools - set(results)

if missing:
    raise RuntimeError(
        f"The model omitted {missing}. Rerun the model-call cell and check its tool_calls."
    )

final_response = llm.invoke([
    HumanMessage(content=user_query),
    response,
    *tool_messages
])

print("\nFINAL HEALTH METRICS:")
print(final_response.content)


calculate_bmi RESULT:
BMI: 22.86


RuntimeError: The model omitted {'calculate_daily_calories'}. Rerun the model-call cell and check its tool_calls.

**Exercise 9 — Smart Meeting Scheduler**

Cell 1 — Create all three tools

In [37]:
from langchain_core.tools import tool

@tool
def check_calendar_availability(person: str, date: str) -> str:
    """Check a person's sample calendar availability on a date in YYYY-MM-DD format."""
    availability = {
        ("Alice", "2026-10-25"): "Available at 10:00 or 14:00.",
        ("Alice", "2026-10-26"): "Available at 11:00."
    }

    return availability.get(
        (person.title(), date),
        f"No sample calendar information found for {person} on {date}."
    )


@tool
def get_timezone_offset(city: str) -> str:
    """Return a sample UTC timezone offset for a city."""
    offsets = {
        "Delhi": "UTC+05:30",
        "Mumbai": "UTC+05:30",
        "London": "UTC+00:00",
        "New York": "UTC-04:00"
    }

    return offsets.get(city.title(), "Timezone not found in sample data.")


@tool
def book_meeting(person: str, date: str, time: str) -> str:
    """Simulate booking a meeting with a person on a date and time."""
    return f"SIMULATED BOOKING: Meeting with {person} on {date} at {time}."

Cell 2 — Bind the tools and ask two questions

In [38]:
tools = [
    check_calendar_availability,
    get_timezone_offset,
    book_meeting
]

tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

availability_query = (
    "Check Alice's calendar availability on 2026-10-25. "
    "Only check availability. Do not book a meeting."
)

booking_query = (
    "Schedule a meeting with Alice on 2026-10-25. "
    "First check her availability, then check the timezone for Delhi, "
    "then book a meeting at one of her available times. "
    "Use all three tools: check_calendar_availability, "
    "get_timezone_offset, and book_meeting."
)

availability_response = llm_with_tools.invoke(availability_query)
booking_response = llm_with_tools.invoke(booking_query)

print(f"{'AVAILABILITY QUERY':<25} | BOOKING QUERY")
print("-" * 75)

availability_names = [c["name"] for c in availability_response.tool_calls]
booking_names = [c["name"] for c in booking_response.tool_calls]

print(f"{str(availability_names):<25} | {booking_names}")

AVAILABILITY QUERY        | BOOKING QUERY
---------------------------------------------------------------------------
['check_calendar_availability'] | ['check_calendar_availability']


Cell 3 — Execute availability-only calls

In [39]:
availability_messages = []

for call in availability_response.tool_calls:
    if call["name"] == "check_calendar_availability":
        result = tool_map[call["name"]].invoke(call["args"])
        print("AVAILABILITY RESULT:", result)

        availability_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

availability_final = llm.invoke([
    HumanMessage(content=availability_query),
    availability_response,
    *availability_messages
])

print("\nAVAILABILITY FINAL ANSWER:")
print(availability_final.content)

AVAILABILITY RESULT: Available at 10:00 or 14:00.

AVAILABILITY FINAL ANSWER:
Alice is free on 2026‑10‑25 at **10:00** or **14:00**.


Cell 4 — Execute booking calls in dependency order

In [40]:
booking_results = {}
booking_messages = []

execution_order = [
    "check_calendar_availability",
    "get_timezone_offset",
    "book_meeting"
]

calls_by_name = {
    call["name"]: call
    for call in booking_response.tool_calls
}

for tool_name in execution_order:
    call = calls_by_name.get(tool_name)

    if call:
        result = tool_map[tool_name].invoke(call["args"])
        booking_results[tool_name] = result

        print(f"{tool_name} RESULT:", result)

        booking_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

missing = set(execution_order) - set(booking_results)

if missing:
    print("\nThe model omitted these requested tools:", missing)
    print("Review the booking tool calls and rerun Cell 2 if needed.")
else:
    booking_final = llm.invoke([
        HumanMessage(content=booking_query),
        booking_response,
        *booking_messages
    ])

    print("\nBOOKING FINAL ANSWER:")
    print(booking_final.content)

check_calendar_availability RESULT: Available at 10:00 or 14:00.

The model omitted these requested tools: {'get_timezone_offset', 'book_meeting'}
Review the booking tool calls and rerun Cell 2 if needed.


**Exercise 10 — Restaurant Recommendation Assistant**

Cell 1 — Create the four tools

In [41]:
from langchain_core.tools import tool

@tool
def find_restaurants(cuisine: str, city: str) -> str:
    """Find sample restaurants matching a cuisine in a city."""
    restaurants = {
        ("italian", "pune"): [
            "Little Italy",
            "Arthur's Theme",
            "Toscano"
        ],
        ("indian", "pune"): [
            "Spice Garden",
            "Deccan Kitchen"
        ]
    }

    matches = restaurants.get(
        (cuisine.lower(), city.lower()),
        []
    )

    return json.dumps(matches if matches else ["No matching sample restaurants found"])


@tool
def get_restaurant_rating(restaurant_name: str) -> str:
    """Get a sample restaurant rating out of 5."""
    ratings = {
        "Little Italy": 4.2,
        "Arthur's Theme": 4.3,
        "Toscano": 4.1
    }

    rating = ratings.get(restaurant_name)
    if rating is None:
        return f"No sample rating found for {restaurant_name}."

    return f"{restaurant_name} sample rating: {rating}/5."


@tool
def check_restaurant_open(restaurant_name: str, day: str) -> str:
    """Check sample opening status of a restaurant on a given day."""
    opening_status = {
        "Little Italy": True,
        "Arthur's Theme": True,
        "Toscano": False
    }

    if restaurant_name not in opening_status:
        return f"No sample opening data for {restaurant_name}."

    status = "open" if opening_status[restaurant_name] else "closed"
    return f"{restaurant_name} is listed as {status} on {day} in the sample data."


@tool
def estimate_cost_for_two(restaurant_name: str) -> str:
    """Estimate sample meal cost in rupees for two people."""
    costs = {
        "Little Italy": 1800,
        "Arthur's Theme": 2000,
        "Toscano": 2200
    }

    cost = costs.get(restaurant_name)
    if cost is None:
        return f"No sample cost data found for {restaurant_name}."

    return f"Estimated sample cost for two at {restaurant_name}: ₹{cost}."

Cell 2 — Bind all four tools and ask the model

In [42]:
tools = [
    find_restaurants,
    get_restaurant_rating,
    check_restaurant_open,
    estimate_cost_for_two
]

tool_map = {t.name: t for t in tools}
llm_with_tools = llm.bind_tools(tools)

user_query = """
Suggest an Italian restaurant in Pune that's open today, with its rating
out of 5 and approximate cost for two.

Use find_restaurants first, choose one matching restaurant, then use
get_restaurant_rating, check_restaurant_open, and estimate_cost_for_two
for that same restaurant. Use all four tools.
"""

response = llm_with_tools.invoke(user_query)

print("MODEL'S SUGGESTED TOOL CALLS:")
for call in response.tool_calls:
    print(call)

MODEL'S SUGGESTED TOOL CALLS:
{'name': 'find_restaurants', 'args': {'city': 'Pune', 'cuisine': 'Italian'}, 'id': 'fc_29f28d52-a77b-45c9-bc4b-00a215ad45fb', 'type': 'tool_call'}


Cell 3 — Execute the suggested tools

In [43]:
tool_messages = []
results = {}

for call in response.tool_calls:
    selected_tool = tool_map.get(call["name"])

    if selected_tool:
        result = selected_tool.invoke(call["args"])
        results[call["name"]] = result

        print(f"\n{call['name']} RESULT:")
        print(result)

        tool_messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"]
            )
        )

required_tools = {
    "find_restaurants",
    "get_restaurant_rating",
    "check_restaurant_open",
    "estimate_cost_for_two"
}

missing = required_tools - set(results)

if missing:
    print("\nThe model omitted these requested tools:", missing)
    print("Review the tool_calls and rerun the model-call cell if needed.")
else:
    final_response = llm.invoke([
        HumanMessage(content=user_query),
        response,
        *tool_messages
    ])

    print("\nFINAL RESTAURANT RECOMMENDATION:")
    print(final_response.content)


find_restaurants RESULT:
["Little Italy", "Arthur's Theme", "Toscano"]

The model omitted these requested tools: {'check_restaurant_open', 'get_restaurant_rating', 'estimate_cost_for_two'}
Review the tool_calls and rerun the model-call cell if needed.
